# Decision Tree Regression
##### Owen Lindsey
##### Professor Majumdar
##### AIT-110: Statistical Learning Theory
##### Grand Canyon University
##### Date: 
##### Video Link: 

# Understanding Decision Trees and Random Forests

## 1. Building a Decision Tree

***Tree Structure*** -- Decision Trees are comprised of a ***root node***, ***intermediate nodes***, and ***leaf nodes***. These trees may look abnormal, due to the root being the top most node, with the rest of the nodes following BELOW and splitting based on deterministic outcomes.

***Choosing Splits*** -- The algorithm works by evaluating all possible splits across all features to find the best one. For each feature, it tests different threshold values and calculates the resulting purity of the child nodes. The split that produces the greatest improvement in purity (or reduction in impurity) is chosen.

***Measuring Purity*** -- For classification, the Gini Index measures impurity, where values closer to 0 indicate purer nodes (0 is completely pure). Information Gain measures how much information was gained by splitting at a specific location. For regression trees, we use variance reduction or MSE (Mean Squared Error) to measure split quality - choosing splits that minimize the variance within each child node.

***Pruning*** -- To prevent overfitting, trees can be pruned using two approaches:
- **Pre-pruning**: Stop growing the tree early by setting constraints (max depth, minimum samples per leaf, minimum impurity decrease)
- **Post-pruning**: Grow a full tree, then remove branches that don't improve performance on validation data using cost-complexity pruning (alpha parameter)

## 2. Limitations of Decision Trees

Decision trees have several key limitations:
- **High Variance**: Small changes in data can result in completely different tree structures, making them unstable
- **Overfitting**: Trees can grow too complex and memorize training data rather than learning generalizable patterns
- **Greedy Algorithm**: Trees make locally optimal decisions at each split, which may not lead to the globally optimal tree
- **Difficulty with Linear Relationships**: Trees struggle to model simple linear relationships efficiently

***How Random Forests Address These*** -- Random Forests tackle these limitations by combining multiple trees trained on different subsets of data with random feature selection. This ensemble approach reduces variance and overfitting while maintaining predictive power.

## 3. Random Forest Algorithm

***Combining Multiple Trees*** -- Random Forests improve model performance by creating an ensemble of decision trees and aggregating their predictions. For regression, predictions are averaged; for classification, the majority vote is used.

***Bootstrapping (Bagging)*** -- Each tree is trained on a bootstrapped sample of the original dataset. This means randomly sampling the data with replacement, so each tree sees a slightly different version of the data (typically ~63% unique samples, with some repeated).

***Random Feature Selection*** -- At each split in each tree, only a random subset of features is considered (typically √p for classification or p/3 for regression, where p is the total number of features). This decorrelates the trees, preventing them from all making the same mistakes.

## 4. Variance Reduction

***How It Works*** -- By averaging predictions across many trees, Random Forests significantly reduce variance compared to a single decision tree. Each individual tree may overfit to its particular bootstrap sample, but these overfitting patterns are different across trees. When averaged, the overfitting cancels out while the true signal remains.

***Benefits for Accuracy*** -- Lower variance means the model generalizes better to unseen data. While individual trees have low bias but high variance, Random Forests maintain that low bias while achieving much lower variance. This leads to more stable predictions and better performance on test data.

## 5. Variable Importance

***Measuring Importance*** -- Random Forests provide two main metrics for variable importance:

1. **Mean Decrease in Impurity (MDI)**: Measures how much each feature decreases the weighted impurity (Gini or MSE) across all trees. Features that create larger decreases in impurity are more important.

2. **Permutation Importance**: Measures how much the model's accuracy decreases when a feature's values are randomly shuffled. Features that cause larger drops in accuracy when permuted are more important.

***Interpretation and Use*** -- Variable importance scores help identify which features contribute most to predictions. This is useful for:
- Feature selection (removing unimportant features)
- Understanding the underlying relationships in the data
- Communicating model insights to stakeholders
- Debugging model behavior 



# 2. Building Decision Trees and Random Forests



In [12]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import *
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score




from IPython.display import Image, display_svg, SVG
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
import pandas as pd
import numpy as np
%matplotlib inline

customer_data = pd.read_csv('data/customer_feedback_satisfaction.csv')

customer_data.head()

customer_data.info()

feedback_map = {'Low': 0, 'Medium': 1, 'High': 2}
loyalty_map = {'Bronze': 0, 'Silver': 1, 'Gold': 2}

customer_data['FeedbackScore'] = customer_data['FeedbackScore'].map(feedback_map)
customer_data['LoyaltyLevel'] = customer_data['LoyaltyLevel'].map(loyalty_map)
customer_data['Gender'] = customer_data['Gender'].map({'Male': 0, 'Female': 1})
customer_data['Country'] = customer_data['Country'].map({'United States': 0, 'Canada': 1, 'Mexico': 2, 'Other': 3})

# drop_first=True avoids multicollinearity (you only need n-1 columns for n categories)
customer_data = pd.get_dummies(customer_data, columns=['Gender', 'Country'], drop_first=True)




X_train, X_test, y_train, y_test = train_test_split(
    customer_data[['Age','Gender','Country','Income','ProductQuality','ServiceQuality','PurchaseFrequency','FeedbackScore','LoyaltyLevel']],
    customer_data['SatisfactionScore'],
    test_size=0.2,
    random_state=42
)

dt_regressor = DecisionTreeRegressor()

dt_regressor.fit(X_train, y_train)

y_pred = dt_regressor.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38444 entries, 0 to 38443
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         38444 non-null  int64  
 1   Age                38444 non-null  int64  
 2   Gender             38444 non-null  object 
 3   Country            38444 non-null  object 
 4   Income             38444 non-null  int64  
 5   ProductQuality     38444 non-null  int64  
 6   ServiceQuality     38444 non-null  int64  
 7   PurchaseFrequency  38444 non-null  int64  
 8   FeedbackScore      38444 non-null  object 
 9   LoyaltyLevel       38444 non-null  object 
 10  SatisfactionScore  38444 non-null  float64
dtypes: float64(1), int64(6), object(4)
memory usage: 3.2+ MB


KeyError: "['Gender', 'Country'] not in index"